# ComplianceRadar: fine-tune the event classifier with LoRA

Trains a small multilingual encoder (`xlm-roberta-base`) with **LoRA (PEFT)** to label regulatory/news headlines with one of 13 topics.

- **Training data:** LLM-distilled headlines (`data/classifier/train.jsonl`, made by `scripts/make_classifier_dataset.py`).
- **Test data:** 52 hand-written headlines (`data/fixtures/classifier_test.jsonl`) the generator never saw.
- **Baselines on the same test set:** keyword rules 0.60 accuracy; rules → LLM 0.92 with 24 LLM calls.
- **Result of this notebook (13 Nov 2026):** 0.808 accuracy alone; as the middle tier (rules → LoRA → LLM) 0.94 with 5 LLM calls.

Runtime: **GPU (T4 is enough)**, about 5 minutes. At the end the adapter can be pushed to the Hugging Face Hub and loaded by the worker.

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "datasets>=2.20" "accelerate>=0.33" scikit-learn
# Colab ships an old torchao that newer peft refuses to import alongside. This notebook does not use torchao.
!pip -q uninstall -y torchao

In [ ]:
# Get the repo (for the data files). Change the URL if your fork differs.
!git clone -q https://github.com/Muskaanmahadik10/compliance-radar.git || (cd compliance-radar && git pull -q)
%cd compliance-radar
!wc -l data/classifier/train.jsonl data/fixtures/classifier_test.jsonl

In [ ]:
import json
import random

import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

train_rows = load("data/classifier/train.jsonl")
test_rows = load("data/fixtures/classifier_test.jsonl")
labels = sorted({r["topic"] for r in train_rows} | {r["topic"] for r in test_rows})
label2id = {l: i for i, l in enumerate(labels)}; id2label = {i: l for l, i in label2id.items()}

# Leakage guard: no test headline may appear in training.
test_titles = {r["title"].lower() for r in test_rows}
train_rows = [r for r in train_rows if r["title"].lower() not in test_titles]
random.shuffle(train_rows)
n_val = max(len(labels) * 4, int(0.1 * len(train_rows)))
val_rows, train_rows = train_rows[:n_val], train_rows[n_val:]
print(len(train_rows), "train /", len(val_rows), "validation /", len(test_rows), "test ·", len(labels), "labels")

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

BASE = "xlm-roberta-base"
tok = AutoTokenizer.from_pretrained(BASE)

def to_ds(rows):
    ds = Dataset.from_list([{"text": r["title"], "label": label2id[r["topic"]]} for r in rows])
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True)

ds_train, ds_val, ds_test = to_ds(train_rows), to_ds(val_rows), to_ds(test_rows)

model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(labels), id2label=id2label, label2id=label2id)
lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["query", "value"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # LoRA trains well under 1.5% of the weights

In [ ]:
def metrics(p):
    pred = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, pred), "macro_f1": f1_score(p.label_ids, pred, average="macro")}

args = TrainingArguments(
    output_dir="clf-out", learning_rate=5e-4, per_device_train_batch_size=32, per_device_eval_batch_size=64,
    num_train_epochs=12, weight_decay=0.01, warmup_ratio=0.1, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1", logging_steps=20, seed=SEED, report_to="none",
    fp16=torch.cuda.is_available(),
)
trainer = Trainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val, data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
# Final check on the hand-written test set (never seen in training or model selection).
out = trainer.predict(ds_test)
pred = out.predictions.argmax(-1)
print("TEST accuracy %.3f · macro-F1 %.3f" % (accuracy_score(out.label_ids, pred), f1_score(out.label_ids, pred, average="macro")))
print(classification_report(out.label_ids, pred, target_names=labels, zero_division=0))
for r, p in zip(test_rows, pred):
    if id2label[int(p)] != r["topic"]:
        print("MISS:", r["topic"], "->", id2label[int(p)], "|", r["title"])

result = {"tier": "lora (xlm-roberta-base, r=16)", "n": len(test_rows), "accuracy": round(float(accuracy_score(out.label_ids, pred)), 3),
          "macro_f1": round(float(f1_score(out.label_ids, pred, average="macro")), 3), "train_rows": len(train_rows), "cost_usd_per_1k_events": 0.0}
json.dump(result, open("lora_classifier_result.json", "w"), indent=2); result

## Save and (optionally) publish the adapter

Paste the numbers from `lora_classifier_result.json` into the README table. To use the model in the worker, push the adapter to the Hub and set `RADAR_CLASSIFIER_MODEL=<your-username>/compliance-radar-event-classifier`.

In [ ]:
model.save_pretrained("clf-adapter"); tok.save_pretrained("clf-adapter")
# from huggingface_hub import notebook_login; notebook_login()
# model.push_to_hub("YOUR_USERNAME/compliance-radar-event-classifier"); tok.push_to_hub("YOUR_USERNAME/compliance-radar-event-classifier")